# Module 1 Exercise: Time Travel on the `orders` Delta Table

Every Delta write adds a commit to the table's transaction log. `DataBuilder`
appended `orders` in **12 batches**, so the table has 12 committed versions on
disk, each representing ~2.3M more rows than the previous one.

This notebook shows you how to:

1. Read the transaction log
2. Query **any historical version** of the table
3. Query by **timestamp** instead of version number


## 1. See the full transaction log

Every commit is listed, newest first. SQL Server analogue: think *backup history*
plus *transaction log*, rolled into one.

In [ ]:
%%sql
DESCRIBE HISTORY orders


## 2. Read the table at a specific version

`VERSION AS OF` rewinds to the state of the table right after commit *N*. The
table file structure on disk is unchanged - Delta just filters the log.

In [ ]:
%%sql
SELECT 'v0  (first commit)' AS label, COUNT(*) AS row_count FROM orders VERSION AS OF 0
UNION ALL
SELECT 'v6 (mid)',               COUNT(*)               FROM orders VERSION AS OF 6
UNION ALL
SELECT 'v10 (last commit)',       COUNT(*)               FROM orders VERSION AS OF 10
UNION ALL
SELECT 'current',                 COUNT(*)               FROM orders


## 4. Time travel by timestamp

`TIMESTAMP AS OF` accepts a timestamp literal and returns the table state as of
that moment. First, pull a timestamp from the log, then paste it into the
literal. This is useful when you know *when* something happened but not
*which commit*.


In [ ]:
%%sql
SELECT MIN(timestamp) AS first_commit,
       MAX(timestamp) AS last_commit
FROM   (DESCRIBE HISTORY orders)


Copy one of the timestamps from the result above into the query below.


In [ ]:
%%sql
-- Replace the literal with a timestamp from the previous cell
SELECT COUNT(*) AS row_count
FROM   orders TIMESTAMP AS OF '2026-04-24 09:00:00'


## Key takeaway

Delta's transaction log gives you *free* point-in-time reads of any committed
version. No backups, no snapshots, no extra storage management. The same log
drives every downstream engine (Warehouse, Power BI Direct Lake), so a version
read in Spark returns the same rows as the same version read in T-SQL.
